# Proyek Akhir: Menyelesaikan Permasalahan Perusahaan Edutech

## Business Understanding

### Latar Belakang
Jaya Jaya Institut merupakan salah satu institusi pendidikan perguruan yang telah berdiri sejak tahun 2000 dan telah mencetak banyak lulusan dengan reputasi yang sangat baik. Namun, institusi ini menghadapi masalah signifikan yaitu masih banyak siswa yang tidak menyelesaikan pendidikannya (dropout).

Tingginya angka dropout menjadi masalah besar yang dapat mempengaruhi:
- Reputasi institusi
- Efisiensi biaya operasional
- Peluang karier graduates
- Rate of return investasi pendidikan

### Permasalahan Bisnis
1. **Tingginya angka dropout** - Proporsi siswa yang tidak menyelesaikan pendidikan cukup tinggi (~32%)
2. **Kurangnya deteksi dini** - Institusi belum memiliki sistem prediktif untuk mengidentifikasi siswa berisiko
3. **Tidak efektifnya intervensi** - Akibat tidak ada deteksi dini, bimbingan khusus tidak dapat diberikan secara tepat waktu

### Cakupan Proyek
Proyek ini mencakup pengembangan:
1. **Analisis Data** - Eksplorasi data untuk memahami karakteristik mahasiswa
2. **Machine Learning** - Model prediktif untuk mendeteksi risiko dropout
3. **Dashboard** - Visualisasi data untuk monitoring performa siswa
4. **Prototype** - Sistem prediksi interaktif berbasis Streamlit

- Nama: Fahru Alfarizi Hananza Putrawan
- Email: fahrualfarizi7@gmail.com
- Id Dicoding: fahrual_19

## Persiapan

### Menyiapkan library yang dibutuhkan

In [ ]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Menampilkan versi library
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: sklearn imported")

### Menyiapkan data yang akan digunakan

In [ ]:
# Memuat dataset
df = pd.read_csv('data.csv', sep=';')

# Menampilkan dimensi dataset
print(f"Dimensi dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
print("\n")

# Menampilkan 5 data pertama
df.head()

## Data Understanding

Pada tahap ini, kita akan melakukan eksplorasi data untuk memahami karakteristik dataset lebih dalam.

In [ ]:
# Informasi dataset
print("Informasi Dataset:")
print("="*50)
df.info()
print("\n")

# Statistik deskriptif
print("\nStatistik Deskriptif:")
print("="*50)
df.describe()

In [ ]:
# Analisis distribusi target variable (Status)
print("\nDistribusi Target Variable (Status):")
print("="*50)
print(df['Status'].value_counts())
print("\n")
print("Persentase:")
print(df['Status'].value_counts(normalize=True) * 100)

# Visualisasi distribusi Status
plt.figure(figsize=(8, 5))
colors = ['#2ecc71', '#e74c3c', '#3498db']
df['Status'].value_counts().plot(kind='bar', color=colors)
plt.title('Distribusi Status Mahasiswa', fontsize=14, fontweight='bold')
plt.xlabel('Status')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
for i, v in enumerate(df['Status'].value_counts()):
    plt.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pengecekan missing values
print("\nMissing Values:")
print("="*50)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing Percentage': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

### Analisis Korelasi dengan Status

Mari kita lihat korelasi antara fitur-fitur numerik dengan target variable.

In [ ]:
# Encode target variable untuk analisis korelasi
le = LabelEncoder()
df['Status_encoded'] = le.fit_transform(df['Status'])
print("Encoding Status: ", dict(zip(le.classes_, le.transform(le.classes_))))

# Korelasi fitur dengan target
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
correlations = df[numeric_cols].corr()['Status_encoded'].sort_values(ascending=False)
print("\nKorelasi dengan Status (encoded):")
print(correlations)

# Visualisasi korelasi fitur penting
plt.figure(figsize=(12, 8))
corr_with_target = correlations.drop('Status_encoded')
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in corr_with_target.values]
corr_with_target.plot(kind='barh', color=colors)
plt.title('Korelasi Fitur dengan Status Mahasiswa', fontsize=14, fontweight='bold')
plt.xlabel('Korelasi')
plt.tight_layout()
plt.show()

## Data Preparation / Preprocessing

Pada tahap ini, kita akan melakukan pembersihan dan persiapan data sebelum modeling.

In [ ]:
# Mengubah target menjadi binary: Dropout (1) vs Non-Dropout (0)
# Graduate dan Enrolled dianggap sebagai "Belum Dropout"
df['Target'] = (df['Status'] == 'Dropout').astype(int)
print("Distribusi Target (Binary):")
print(df['Target'].value_counts())
print(f"\nProporsi Dropout: {df['Target'].mean()*100:.2f}%")

# Menghapus kolom Status dan Status_encoded (sudah di-encode ke Target)
df_model = df.drop(['Status', 'Status_encoded'], axis=1)
print(f"\nDataset siap untuk modeling: {df_model.shape}")

In [ ]:
# Memisahkan fitur dan target
X = df_model.drop('Target', axis=1)
y = df_model['Target']

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data Training: {X_train.shape[0]} sampel")
print(f"Data Testing: {X_test.shape[0]} sampel")
print(f"\nDistribusi target training:")
print(y_train.value_counts())
print(f"\nDistribusi target testing:")
print(y_test.value_counts())

In [ ]:
# Feature Scaling menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature Scaling dilakukan!")
print(f"Shape X_train_scaled: {X_train_scaled.shape}")
print(f"Shape X_test_scaled: {X_test_scaled.shape}")

## Modeling

Pada tahap ini, kita akan membangun model Machine Learning untuk memprediksi kemungkinan dropout mahasiswa.

In [ ]:
# Membangun model Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# Training model
print("Melatih model Random Forest...")
rf_model.fit(X_train_scaled, y_train)
print("Model berhasil dilatih!")

# Prediksi pada data testing
y_pred = rf_model.predict(X_test_scaled)

# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Fitur Paling Penting:")
print(feature_importance.head(10))

In [ ]:
# Visualisasi Feature Importance
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(top_features)))[::-1]
plt.barh(range(len(top_features)), top_features['importance'].values, color=colors)
plt.yticks(range(len(top_features)), top_features['feature'].values)
plt.xlabel('Importance')
plt.ylabel('Fitur')
plt.title('Top 15 Fitur Penting untuk Prediksi Dropout', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nInsight:")
print("- Fitur 'Curricular_units_approved' (satuan kurikulum yang disetujui) adalah")
print("  fitur paling penting dalam memprediksi dropout.")
print("- Fitur lain yang juga penting: nilai, usia, dan status pembayaran tuition.")

## Evaluation

Pada tahap ini, kita akan mengevaluasi performa model yang telah dibangun.

In [ ]:
# Evaluasi Model
print("="*60)
print("EVALUASI MODEL RANDOM FOREST")
print("="*60)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy*100:.2f}%")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-Dropout', 'Dropout']))

# F1 Score
f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1*100:.2f}%")

# Confusion Matrix Visualization
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non-Dropout', 'Dropout'],
            yticklabels=['Non-Dropout', 'Dropout'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.tight_layout()
plt.show()

print("\nInterpretasi:")
print(f"- True Negatives (Non-Dropout benar): {cm[0][0]}")
print(f"- False Positives (Salah prediksi Dropout): {cm[0][1]}")
print(f"- False Negatives (Salah prediksi Non-Dropout): {cm[1][0]}")
print(f"- True Positives (Dropout benar): {cm[1][1]}")

In [ ]:
# Menyimpan model dan scaler untuk deployment
import joblib
import os

# Membuat folder model jika belum ada
os.makedirs('model', exist_ok=True)

# Simpan model
joblib.dump(rf_model, 'model/rf_model.joblib')
print("Model berhasil disimpan di 'model/rf_model.joblib'")

# Simpan scaler
joblib.dump(scaler, 'model/scaler.joblib')
print("Scaler berhasil disimpan di 'model/scaler.joblib'")

# Simpan feature names
joblib.dump(list(X.columns), 'model/feature_names.joblib')
print("Feature names berhasil disimpan di 'model/feature_names.joblib'")

# Simpan label encoder
joblib.dump(le, 'model/label_encoder.joblib')
print("Label encoder berhasil disimpan di 'model/label_encoder.joblib'")

print("\nSemua artefak model siap untuk deployment!")